### Middleware

Middleware provides a way to more tightly control what happens inside the agent.Middleware is useful for the following:
    
    * Tracking agent behaviour with logging analystics and debugging.
    * Transforming prompts tool selection, and output formatting
    * Adding retries, fallbacks and early termination logic
    * Applying rate limits,quadrails and PII detection

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
from langchain.chat_models import init_chat_model
model = init_chat_model(
     "google/gemma-4-26b-a4b-it:free",
        model_provider="openrouter"
)
model

d:\LangchainV2\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17', 'langchain-openrouter': '0.2.8'}}, profile={'name': 'Gemma 4 26B A4B  (free)', 'release_date': '2026-04-02', 'last_updated': '2026-04-02', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001FFB0D9AA50>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='google/gemma-4-26b-a4b-it:free', model_kwargs={})

In [4]:
from langchain.chat_models import init_chat_model
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
model_hugging = init_chat_model(
    "gpt-4o",  # or "gpt-3.5-turbo"
    model_provider="openai"
)

### Summarization

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context.Summarization is useful for the following:
* Long running conversations that exceeded context windows.
* Multi turn dialogues with extensive history.
* Applications where preserving full conversation context matters.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import  SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage


create_agent=create_agent(model=model_hugging,checkpointer=InMemorySaver(),middleware=[
    SummarizationMiddleware(model=model_hugging,
                            trigger=("messages",10),
                            keep=("messages",4))]
                            )

In [4]:
### Run with thread id

config={"configurable":{"thread_id":"test-1"}}

In [5]:
questions=[
    "What is 2+2?",
    "what is 10*5??",
    "what is 100/4?",
    "What is 15-7?",
    "what is 3*3?",
    "what is 4*4?"
    ]

for q in questions:
    response=create_agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='33fedc3b-81ea-4005-96e3-79c70948230f'), AIMessage(content='4', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 14, 'total_tokens': 15, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-ENIC8Zpg0yxCPn1WmtHYDpssI8lJb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a095d5-7a0c-7722-8b0d-838cfef1a12c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 1, 'total_tokens': 15, 'input_token_details': 

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver



@tool
def search_hotel(city:str)-> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1: Grand Hotel -5 star, $350/night,spa,pool, gym
    2: City Inn - 4 star,$180/night,business centre
    3: Budget stay - 3 star, $75/night,free wifi
    """

create_agent = create_agent(model=model_hugging,
                            tools=[search_hotel],checkpointer=InMemorySaver(),
                            middleware=[
                                SummarizationMiddleware(
                                    model=model_hugging,
                                    trigger=("tokens",550),
                                    keep=("tokens",200),
                                    )
                                    ]
                                    )


config= {"configurable": {"thread_id":"test-1"}}

def count_tokens(messges):
    total_chars = sum(len(str(m.content)) for m in messges)
    return total_chars //4 #4 character equals 1

In [8]:
cities= ["Paris","London","Tokyo","New York","Dubai"]

for city in cities:
    response = create_agent.invoke({"messages":[HumanMessage(content=f"find hotels in {city}")]},config=config)
    tokens = count_tokens(response["messages"])
    print(f"{city} : ~{tokens} token,{len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris : ~149 token,4 messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='f41c9827-8eca-4333-b76c-0331289779fe'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 55, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-ENICkc9YkF19qxnqaBdLMxLb8M3sP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a095d6-14c2-77d3-aaf1-11a28a2ede05-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'call_R9Ebv8yCPj0zytcnuGwSAbv1', 'type': 'tool

### Fraction

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver



@tool
def search_hotel(city:str)-> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1: Grand Hotel -5 star, $350/night,spa,pool, gym
    2: City Inn - 4 star,$180/night,business centre
    3: Budget stay - 3 star, $75/night,free wifi
    """

create_agent = create_agent(model=model_hugging,
                            tools=[search_hotel],checkpointer=InMemorySaver(),
                            middleware=[
                                SummarizationMiddleware(
                                    model=model_hugging,
                                    trigger=("fraction",0.005), #0.5%  = ~640 tokens
                                    keep=("fraction",0.002), # 0.2% = ~256 tokens
                                    )
                                    ]
                                    )


config= {"configurable": {"thread_id":"test-1"}}

def count_tokens(messges):
    total_chars = sum(len(str(m.content)) for m in messges)
    return total_chars //4 #4 character equals 1



cities= ["Paris","London","Tokyo","New York","Dubai"]

for city in cities:
    response = create_agent.invoke({"messages":[HumanMessage(content=f"find hotels in {city}")]},config=config)
    tokens = count_tokens(response["messages"])
    fraction = tokens/128000 
    print(f"{city} : ~{tokens} token  ({fraction: .4%}), {len(response['messages'])} messages")

    print(f"{(response['messages'])}")



Paris : ~124 token  ( 0.0969%), 4 messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='ee0f4eec-ba36-4ffc-95cf-78c67954e982'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c9a0e786b8', 'id': 'chatcmpl-ENVAlHpG2rvlpB3eFkkiCtimPW1CD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a098ce-b884-70d1-86d7-8f48d9df59cf-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'call_3YOW79slf

### Human in the loop MiddleWare



In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def read_email_tool(email_id: str)-> str:
    """Mock function to read an email by its ID."""
    return f"Email sent for mail id: {email_id}"


@tool
def send_email_tool(recipient : str, subject: str, body: str)-> str:
    """Mock Function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body: '{body}' "

In [23]:
agent = create_agent(model=model_hugging,
                     tools=[send_email_tool,read_email_tool],
                     checkpointer=InMemorySaver(),
                     middleware=[HumanInTheLoopMiddleware(interrupt_on={
                    "send_email_tool":{
                        "allowed_decisions":["approve","edit","reject"]
                    },"read_email_tool":False,
})])

In [25]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke({"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?' ")]},config=config)

OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_ewXDDV9sLvVdP2fQQ5PTVjAq", 'type': 'invalid_request_error', 'param': 'messages.[2].role', 'code': None}}

In [10]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='9bbeb52a-19e2-44e1-a35a-1d709a9b94a9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 98, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c9a0e786b8', 'id': 'chatcmpl-ENVB78NBNoQVGc7b3mmpvDfnCloqk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a098cf-0e42-7f20-997c-fc9d1385496a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient

In [17]:
from langgraph.types import Command


if "__interrupt__" in result:
    print(" Paused! Approve....")

    
    result = agent.invoke(
        Command(resume={"decisions":[{"type":"reject"}]}
               ), config=config
    )

print(f"result: {result['messages'][-1].content}")

 Paused! Approve....
result: It seems that the email wasn't sent. If you need further assistance or decide to proceed, feel free to let me know!


In [20]:
from langgraph.types import Command


result = agent.invoke({"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?' ")]},config=config)





In [ ]:

if "__interrupt__" in result:
    print(" Paused! Editing....")

    
    result = agent.invoke(
        Command(resume={"decisions":[{"type":"edit","edited_action":{
            "name":"send_email_tool",
            "args":{
                "recipient":"correct@email.com",
                "subject":"corrected subject",
                "body":" this was edited by human before sending"
            }
        }}]}
               ), config=config
    )

print(f"result: {result['messages'][-1].content}")
result

 Paused! Editing....


OpenAIInvalidRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_eFS5pP53V6oSz0BBFkSUBwOM", 'type': 'invalid_request_error', 'param': 'messages.[6].role', 'code': None}}